In [ ]:
from transformers import AutoModelForCausalLM
import torch.nn as nn
import torch
from peft import LoftQConfig, LoraConfig, get_peft_model, TaskType

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B")
print(model)

In [ ]:

# print(model.config)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # target_modules=[""]
)

base_model = get_peft_model(base_model, lora_config)
base_model.print_trainable_parameters()

class FinetunedEBM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = self.base_model.config.hidden_size
        self.energy_head = nn.Linear(self.hidden_size, 1)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden = outputs.hidden_states[:,-1,:] # last token
        return energy_head(hidden)
    
        
model = FinetunedEBM(base_model)

# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(f"Trainable: {name}")



Loading weights: 100%|██████████| 399/399 [00:00<00:00, 450.68it/s, Materializing param=model.norm.weight]                              


trainable params: 7,667,712 || all params: 8,198,403,072 || trainable%: 0.0935


In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
tokenizer.pad_token = tokenizer.eos_token


class LoraTrainer():
    def __init__(self, model, optim, loss_fn):
        self.model = model
        self.optim = optim
        self.loss_fn = loss_fn
        self.device = "cuda"

        # TODO
    def train(self, train_loader, num_epochs=1, print_every=10):
        self.model.train()
        global_step = 0

        for epoch in range(num_epochs):
            for batch in tqdm(train_loader):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = 

# optimizer with separate LRs
optimizer = torch.optim.AdamW([
    {"params": [p for p in model.base_model.parameters() if p.requires_grad], "lr": 1e-4},
    {"params": model.energy_head.parameters(), "lr": 1e-3},
])

loss_fn = nn.CrossEntropyLoss()
device = torch.device("cuda")
model.to(device)

# train loop
model.train()

for epoch in range(num_epochs):
    total_loss = 0

    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
    
        logits = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)